In [1]:
import json
import torch 
import murmurhash
from functools import partial
import sys
sys.path.append("../")
from tiger.modeling.models import TigerModel, CorrectItemsLogitsProcessor
print(torch.__version__)

/Users/sartq/miniconda3/envs/tiger/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.12.0


In [2]:
config_path = "../tiger/configs/tiger_train_config.json"
index_path = "../data/Beauty/index_rqvae.json" 
interaction_path = "../data/Beauty/inter.json"

def read_json(path):
    with open(path, "r") as f:
        json_file = json.load(f)
    return json_file

config_json = read_json(config_path)
index_json = read_json(index_path)
interaction_json = read_json(interaction_path)

print("config json:", config_json)
print("index json:", index_json) 
print("interation json:", interaction_json)


config json: {'experiment_name': 'tiger_beauty', 'dataset': {'inter_json_path': '../data/Beauty/inter.json', 'index_json_path': '../data/Beauty/index_rqvae.json', 'num_codebooks': 4, 'max_sequence_length': 20, 'sampler_type': 'tiger'}, 'dataloader': {'train_batch_size': 256, 'validation_batch_size': 64}, 'model': {'embedding_dim': 128, 'codebook_size': 256, 'num_positions': 20, 'user_ids_count': 2000, 'num_heads': 4, 'num_encoder_layers': 2, 'num_decoder_layers': 2, 'dim_feedforward': 512, 'd_kv': 32, 'dropout': 0.1, 'activation': 'relu', 'num_beams': 20, 'top_k': 20, 'layer_norm_eps': 1e-06, 'initializer_range': 0.02}, 'optimizer': {'lr': 0.0003}, 'early_stopping_threshold': 20}
index json: {'0': [228, 220, 159, 91], '1': [144, 214, 104, 175], '2': [56, 129, 156, 158], '3': [56, 87, 12, 173], '4': [17, 147, 234, 10], '5': [33, 80, 174, 255], '6': [157, 98, 230, 215], '7': [103, 103, 87, 143], '9': [178, 250, 155, 139], '8': [119, 226, 144, 6], '10': [228, 102, 165, 252], '13': [49, 10

In [3]:
for k, v in config_json.items():
    print(k,": ", v)
print("interaction of user number 33 with items: ", interaction_json["33"]) # interaction of user number 33 with items
print("discrete codebook vectors from RQVAE: ", index_json["700"]) # discrete codebook vectors from RQVAE. 

experiment_name :  tiger_beauty
dataset :  {'inter_json_path': '../data/Beauty/inter.json', 'index_json_path': '../data/Beauty/index_rqvae.json', 'num_codebooks': 4, 'max_sequence_length': 20, 'sampler_type': 'tiger'}
dataloader :  {'train_batch_size': 256, 'validation_batch_size': 64}
model :  {'embedding_dim': 128, 'codebook_size': 256, 'num_positions': 20, 'user_ids_count': 2000, 'num_heads': 4, 'num_encoder_layers': 2, 'num_decoder_layers': 2, 'dim_feedforward': 512, 'd_kv': 32, 'dropout': 0.1, 'activation': 'relu', 'num_beams': 20, 'top_k': 20, 'layer_norm_eps': 1e-06, 'initializer_range': 0.02}
optimizer :  {'lr': 0.0003}
early_stopping_threshold :  20
interaction of user number 33 with items:  [5305, 23, 10453, 925, 1989]
discrete codebook vectors from RQVAE:  [167, 166, 107, 118]


In [4]:
num_codebooks = config_json['dataset']['num_codebooks']
logits_processor_cls = partial(
    CorrectItemsLogitsProcessor,
    num_codebooks=num_codebooks,
    codebook_size=config_json['model']['codebook_size'],
    index_path=index_path,
    num_beams=config_json['model']['num_beams'] # no idea what is this - something related to the beam size search
)

print(logits_processor_cls)

functools.partial(<class 'tiger.modeling.models.tiger.CorrectItemsLogitsProcessor'>, num_codebooks=4, codebook_size=256, index_path='../data/Beauty/index_rqvae.json', num_beams=20)


In [5]:
device = "cpu"
if torch.cuda.is_available():
    device = "cuda"

In [6]:
model = TigerModel(
        embedding_dim=config_json['model']['embedding_dim'],
        codebook_size=config_json['model']['codebook_size'],
        sem_id_len=num_codebooks,
        user_ids_count=config_json['model']['user_ids_count'],
        num_positions=config_json['model']['num_positions'],
        num_heads=config_json['model']['num_heads'],
        num_encoder_layers=config_json['model']['num_encoder_layers'],
        num_decoder_layers=config_json['model']['num_decoder_layers'],
        dim_feedforward=config_json['model']['dim_feedforward'],
        num_beams=config_json['model']['num_beams'],
        num_return_sequences=config_json['model']['top_k'],
        activation=config_json['model']['activation'],
        d_kv=config_json['model']['d_kv'],
        dropout=config_json['model']['dropout'],
        layer_norm_eps=config_json['model']['layer_norm_eps'],
        initializer_range=config_json['model']['initializer_range'],
        logits_processor=logits_processor_cls,
    ).to(device=device)

print(model.dummy_param.device)
print(model)

cpu
TigerModel(
  (model): T5ForConditionalGeneration(
    (shared): Embedding(3034, 128)
    (encoder): T5Stack(
      (embed_tokens): Embedding(3034, 128)
      (block): ModuleList(
        (0): T5Block(
          (layer): ModuleList(
            (0): T5LayerSelfAttention(
              (SelfAttention): T5Attention(
                (q): Linear(in_features=128, out_features=128, bias=False)
                (k): Linear(in_features=128, out_features=128, bias=False)
                (v): Linear(in_features=128, out_features=128, bias=False)
                (o): Linear(in_features=128, out_features=128, bias=False)
                (relative_attention_bias): Embedding(32, 4)
              )
              (layer_norm): T5LayerNorm()
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (1): T5LayerFF(
              (DenseReluDense): T5DenseActDense(
                (wi): Linear(in_features=128, out_features=512, bias=False)
                (wo): Linear(in_features

In [7]:
checkpoint_path = f"../checkpoints/{config_json['experiment_name']}_final_state.pth"

model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()
print(f"Loaded weights from {checkpoint_path}")

Loaded weights from ../checkpoints/tiger_beauty_final_state.pth


In [8]:
user_id = "33"
max_seq_len = 20
num_codebooks = 4
user_id_count = 2000

item_sequences = interaction_json[user_id]
input_items = item_sequences[-max_seq_len:][:-1]
visited = item_sequences[:-1]

flat_semantic_ids = [token for item_id in input_items for token in index_json[str(item_id)]]
hashed_user_id = murmurhash.hash(user_id) % user_id_count

input_data = {
    "semantic_item.ids":    torch.tensor(flat_semantic_ids, dtype=torch.long).to(device),
    "semantic_item.length": torch.tensor([len(flat_semantic_ids)], dtype=torch.long).to(device),
    "hashed_user.ids":      torch.tensor([hashed_user_id], dtype=torch.long).to(device),
    "visited.ids":          torch.tensor(visited, dtype=torch.long).to(device),
    "visited.length":       torch.tensor([len(visited)], dtype=torch.long).to(device),
}


In [11]:
index_json["1989"]

[163, 217, 35, 41]

In [9]:
model.eval()
with torch.inference_mode():
    output = model(input_data)

print(output)

{'predictions': tensor([[[ 229,  271,  606,  831],
         [ 116,  345,  612,  989],
         [ 152,  336,  655, 1013],
         [ 163,  306,  733,  934],
         [  29,  357,  606,  785],
         [  29,  307,  719,  829],
         [  70,  357,  756,  850],
         [ 116,  447,  749,  811],
         [ 149,  489,  715,  840],
         [ 191,  306,  634,  949],
         [  70,  324,  727, 1015],
         [ 159,  491,  532,  913],
         [ 255,  377,  761,  940],
         [ 145,  490,  662,  793],
         [  70,  271,  519,  960],
         [ 116,  492,  662,  788],
         [ 203,  441,  556,  924],
         [ 219,  501,  516,  837],
         [ 229,  271,  575,  903],
         [ 144,  284,  571,  953]]])}
